In [14]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

aliens = pd.read_csv("../data/ben10_aliens.csv")
battles = pd.read_csv("../data/ben10_battles.csv")
enemies = pd.read_csv("../data/ben10_enemies.csv")



In [15]:
aliens.head()

,enemy_id,alien_name,enemy_name
0,1,Alien X,Ultimate Big Chill
1,2,Alien X,Ultimate Swampfire
2,3,AmpFibian,Way Big
3,4,Armodrillo,Astrodactyl
4,5,Astrodactyl,NRG


In [16]:
battles.head()

,battle_id,alien_name,enemy_name,battle_date,winner
0,1,Rath,Eatle,01-01-2023,Rath
1,2,Lodestar,Wildmutt,31-10-2022,Lodestar
2,3,Walkatrout,Grey Matter,04-07-2024,Grey Matter
3,4,Four Arms,Wildmutt,07-10-2024,Wildmutt
4,5,Shock Rock,Eatle,10-07-2022,Eatle


In [17]:
enemies.head()

,enemy_id,alien_name,enemy_name
0,1,Alien X,Ultimate Big Chill
1,2,Alien X,Ultimate Swampfire
2,3,AmpFibian,Way Big
3,4,Armodrillo,Astrodactyl
4,5,Astrodactyl,NRG


In [19]:
aliens.describe().round(2)

,alien_id,strength_level,speed_level,intelligence
count,74.00,74.00,74.00,74.00
mean,37.50,7.72,6.78,7.65
std,21.51,1.72,2.13,1.72
min,1.00,5.00,4.00,5.00
25%,19.25,6.25,5.00,6.00
50%,37.50,8.00,7.00,8.00
75%,55.75,9.00,8.75,9.00
max,74.00,10.00,10.00,10.00


In [21]:
battles_merged = battles.merge(
    aliens,
    how = "left",
    left_on = "alien_name",
    right_on = "alien_name",
    suffixes = ("", "_A")
)

battles_merged = battles_merged.rename(columns = {
    "strength_level": "A_strength",
    "speed_level": "A_speed",
    "intelligence": "A_intelligence"
})
battles_merged = battles_merged.drop(
    columns=["alien_id", "species", "home_planet"]
)


In [22]:
battles_merged = battles_merged.merge(
    aliens,
    how="left",
    left_on="enemy_name",
    right_on="alien_name",
    suffixes=("", "_B")
)
battles_merged = battles_merged.rename(columns={
    "strength_level": "B_strength",
    "speed_level": "B_speed",
    "intelligence": "B_intelligence"
})
battles_merged = battles_merged.drop(
    columns=["alien_id", "species", "home_planet", "alien_name_B"]
)
battles_merged.head()


,battle_id,alien_name,enemy_name,battle_date,winner,A_strength,A_speed,A_intelligence,B_strength,B_speed,B_intelligence
0,1,Rath,Eatle,01-01-2023,Rath,5,8,8,5,7,5
1,2,Lodestar,Wildmutt,31-10-2022,Lodestar,6,8,10,8,6,10
2,3,Walkatrout,Grey Matter,04-07-2024,Grey Matter,5,9,9,8,4,6
3,4,Four Arms,Wildmutt,07-10-2024,Wildmutt,8,4,5,8,6,10
4,5,Shock Rock,Eatle,10-07-2022,Eatle,7,6,6,5,7,5


In [23]:
battles_merged["A_wins"] = (
    battles_merged["winner"] == battles_merged["alien_name"]
).astype(int)


In [24]:
battles_merged["strength_diff"] = battles_merged["A_strength"] - battles_merged["B_strength"]
battles_merged["speed_diff"] = battles_merged["A_speed"] - battles_merged["B_speed"]
battles_merged["intelligence_diff"] = battles_merged["A_intelligence"] - battles_merged["B_intelligence"]


In [25]:
battles_merged["strength_gap"] = battles_merged["strength_diff"].abs()
battles_merged["speed_gap"] = battles_merged["speed_diff"].abs()
battles_merged["intelligence_gap"] = battles_merged["intelligence_diff"].abs()


In [26]:
battles_merged["strength_ratio"] = battles_merged["A_strength"] / (battles_merged["B_strength"] + 1)
battles_merged["speed_ratio"] = battles_merged["A_speed"] / (battles_merged["B_speed"] + 1)
battles_merged["intelligence_ratio"] = battles_merged["A_intelligence"] / (battles_merged["B_intelligence"] + 1)


In [27]:
battles_merged["power_diff"] = (
    (battles_merged["A_strength"] + battles_merged["A_speed"]) -
    (battles_merged["B_strength"] + battles_merged["B_speed"])
)

battles_merged["total_diff"] = (
    (battles_merged["A_strength"] + battles_merged["A_speed"] + battles_merged["A_intelligence"]) -
    (battles_merged["B_strength"] + battles_merged["B_speed"] + battles_merged["B_intelligence"])
)


In [28]:
battles_merged["A_max_stat"] = battles_merged[["A_strength", "A_speed", "A_intelligence"]].max(axis=1)
battles_merged["B_max_stat"] = battles_merged[["B_strength", "B_speed", "B_intelligence"]].max(axis=1)
battles_merged["max_stat_diff"] = battles_merged["A_max_stat"] - battles_merged["B_max_stat"]

battles_merged["A_min_stat"] = battles_merged[["A_strength", "A_speed", "A_intelligence"]].min(axis=1)
battles_merged["B_min_stat"] = battles_merged[["B_strength", "B_speed", "B_intelligence"]].min(axis=1)
battles_merged["min_stat_diff"] = battles_merged["A_min_stat"] - battles_merged["B_min_stat"]


In [29]:
battles_merged["A_variance"] = battles_merged[["A_strength", "A_speed", "A_intelligence"]].var(axis=1)
battles_merged["B_variance"] = battles_merged[["B_strength", "B_speed", "B_intelligence"]].var(axis=1)
battles_merged["variance_diff"] = battles_merged["A_variance"] - battles_merged["B_variance"]


In [30]:
battles_merged["speed_advantage"] = (battles_merged["speed_diff"] > 0).astype(int)
battles_merged["speed_dominance"] = battles_merged["speed_diff"] * battles_merged["speed_ratio"]


In [34]:
drop_cols = [
    "battle_id", "alien_name", "enemy_name",
    "battle_date", "winner",
    "A_strength", "A_speed", "A_intelligence",
    "B_strength", "B_speed", "B_intelligence",
    "A_max_stat", "B_max_stat",
    "A_min_stat", "B_min_stat",
    "A_variance", "B_variance"
]

final_df = battles_merged.drop(columns=drop_cols)

final_df.shape

(300, 17)

In [33]:
final_df.head()

,A_wins,strength_diff,speed_diff,intelligence_diff,strength_gap,speed_gap,intelligence_gap,strength_ratio,speed_ratio,intelligence_ratio,power_diff,total_diff,max_stat_diff,min_stat_diff,variance_diff,speed_advantage,speed_dominance
0,1,0,1,3,0,1,3,0.833333,1.000000,1.333333,1,4,1,0,1.666667,1,1.000000
1,1,-2,2,0,2,2,0,0.666667,1.142857,0.909091,0,0,0,0,0.000000,1,2.285714
2,0,-3,5,3,3,5,3,0.555556,1.800000,1.285714,2,5,1,1,1.333333,1,9.000000
3,0,0,-2,-5,0,2,5,0.888889,0.571429,0.454545,-2,-7,-2,-2,0.333333,0,-1.142857
4,0,2,-1,1,2,1,1,1.166667,0.750000,1.000000,1,2,0,1,-1.000000,0,-0.750000


In [37]:
final_df.to_csv("../data/processed/battle_features.csv", index=False)
